In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

tts = pd.read_csv(DATA_DIR / 'TTS_LBNL_public_file_29-Sep-2025_all.csv', low_memory=False)
tts['installation_date'] = pd.to_datetime(tts['installation_date'], errors='coerce')
tts['year'] = tts['installation_date'].dt.year
print(f'Loaded: {len(tts):,} rows x {len(tts.columns)} cols')

In [ ]:
# Filters — re-run this cell without reloading the data
df = tts[
    tts['year'].isin([2022, 2023, 2024]) &
    (tts['customer_segment'].isin(['RES_SF', 'RES'])) &
    (tts['expansion_system'] == False) &
    (tts['technology_type'].isin(['pv-only', 'pv+storage']))
].copy()

print(f'After filters: {len(df):,} rows')
print(df['year'].value_counts().sort_index().to_string())

In [ ]:
# Group by texas/non-texas, technology_type, year
df['geography'] = df['state'].apply(lambda x: 'Texas' if x == 'TX' else 'Rest of US')

summary = (
    df.groupby(['geography', 'technology_type', 'year'])
    .agg(
        count             = ('PV_system_size_DC', 'count'),
        med_pv_size_dc_kw = ('PV_system_size_DC', 'median'),
        med_battery_kw    = ('battery_rated_capacity_kW', 'median'),
    )
    .round(2)
    .reset_index()
    .sort_values(['geography', 'technology_type', 'year'])
)

display(summary)

out_path = OUTPUT_DIR / 'tx_system_size_summary.csv'
summary.to_csv(out_path, index=False)
print(f'\nSaved to {out_path}')

In [9]:
# All states, 2024 only — pivot by technology_type to spot anomalies in PV size
df_2024 = df[df['year'] == 2024]

by_state = df_2024.groupby(['state', 'technology_type']).agg(
    med_pv_kw  = ('PV_system_size_DC', 'median'),
    med_batt_kw= ('battery_rated_capacity_kW', 'median'),
    n          = ('PV_system_size_DC', 'count'),
).round(2).unstack('technology_type')

# Flatten columns
by_state.columns = [f'{tech}_{metric}' for metric, tech in by_state.columns]
by_state = by_state[['pv-only_med_pv_kw', 'pv-only_med_batt_kw', 'pv+storage_med_pv_kw', 'pv+storage_med_batt_kw',
                      'pv-only_n', 'pv+storage_n']].reset_index()
by_state = by_state.sort_values('pv+storage_med_pv_kw', ascending=False)

display(by_state)

,state,pv-only_med_pv_kw,pv-only_med_batt_kw,pv+storage_med_pv_kw,pv+storage_med_batt_kw,pv-only_n,pv+storage_n
17,TX,9.23,-1.00,15.27,-1.00,"18,877.00","5,242.00"
10,MN,8.08,-1.00,11.55,-1.00,570.00,20.00
3,CT,7.31,-1.00,11.50,10.00,"1,819.00",14.00
5,FL,9.66,-1.00,11.39,-1.00,"1,111.00",82.00
15,OR,8.85,-1.00,11.31,-1.00,312.00,71.00
0,AZ,9.02,-1.00,10.66,10.50,"16,588.00","2,014.00"
7,MA,10.20,-1.00,10.25,7.68,141.00,89.00
6,IL,9.20,-1.00,10.00,-1.00,"19,713.00",550.00
9,ME,8.53,-1.00,10.00,7.60,"3,440.00",27.00
16,RI,8.40,-1.00,9.75,-1.00,201.00,36.00
